# 01 - Explore the evaluation dataset & schema

Owner: Asad. Profiles the curated golden set, the synthetic stress set, and the
schema descriptions. Runnable now against the provisional schema; re-run once the
real DDL + seed data land to profile the seed CSVs too.

In [ ]:
import json, sys, collections
from pathlib import Path
import yaml
ROOT = Path.cwd().parents[0] if Path.cwd().name=='notebooks' else Path.cwd()
sys.path.insert(0, str(ROOT/'src'))
DATA = ROOT/'evaluation'/'datasets'
print('repo root:', ROOT)

## Golden set: tier & pattern coverage

In [ ]:
rows=[json.loads(l) for l in open(DATA/'golden_queries.jsonl') if l.strip()]
print('golden questions:', len(rows))
tiers=collections.Counter(r['difficulty'] for r in rows)
print('tiers:', dict(tiers))
tags=collections.Counter(t for r in rows for t in r.get('tags',[]))
print('pattern coverage:')
for t,c in tags.most_common(): print(f'  {t:24} {c}')

In [ ]:
# bar chart of tiers (matplotlib optional)
try:
    import matplotlib.pyplot as plt
    plt.bar(list(tiers.keys()), list(tiers.values()))
    plt.title('Golden set by difficulty'); plt.ylabel('questions'); plt.show()
except Exception as e:
    print('matplotlib not available:', e)

## Synthetic stress set

In [ ]:
p=DATA/'synthetic_queries.jsonl'
if p.exists():
    syn=[json.loads(l) for l in open(p) if l.strip()]
    print('synthetic queries:', len(syn))
    print('by source:', dict(collections.Counter(s['source'] for s in syn)))
    print('edge cases:', sum(1 for s in syn if 'edge-case' in s.get('tags',[])))
else:
    print('run scripts/generate_synthetic_queries.py first')

## Schema descriptions

In [ ]:
sd=yaml.safe_load(open(ROOT/'db'/'schema_descriptions.yaml'))
for tname,tbl in sd.get('tables',{}).items():
    cols=tbl.get('columns',{})
    todo=sum(1 for c in cols.values() if c.get('description')=='TODO')
    print(f"{tname}: {len(cols)} columns, {todo} TODO descriptions")
    for cn,c in cols.items():
        flags=[k for k in ('pk','fk','currency','timezone','units') if c.get(k)]
        print(f"   - {cn} ({c.get('type','?')}) {('['+','.join(flags)+']') if flags else ''}")

## (Later) Profile the seed CSVs

In [ ]:
seed=ROOT/'db'/'seed'
csvs=sorted(seed.glob('*.csv'))
if not csvs:
    print('No seed CSVs yet - populate db/seed/ then re-run for row/null profiling.')
else:
    import csv as _csv
    for f in csvs:
        rows=list(_csv.reader(open(f)))
        print(f'{f.name}: {len(rows)-1} data rows, {len(rows[0]) if rows else 0} columns')